# Polygon-Aware Agricultural Field Segmentation

This notebook explains the project components. The main idea is to compare a mask-only U-Net baseline against a dual-head U-Net that predicts both field masks and signed distance field (SDF) maps. The SDF head gives the shared model features an explicit boundary-aware learning signal for touching agricultural fields.

## Project Structure

- `cfgs/field_segmentation.py`: experiment dictionaries and selected FTW countries
- `src/data_loaders/`: synthetic and real FTW data modules
- `src/models/unet.py`: mask-only and dual-head U-Net models
- `src/losses/segmentation_losses.py`: BCE, Dice, SDF, boundary, and smoothness losses
- `src/metrics/segmentation_metrics.py`: mIoU, Dice, Boundary IoU, Instance F1, PQ approximation
- `src/trainers/field_trainer.py`: training loop with logs, validation, early stopping, and checkpoints
- `Visualizations/`, `Logs/`, `Saved/`: generated figures, training logs, and model checkpoints

In [ ]:
from pathlib import Path
import os
import sys

def find_project_root(start):
    for path in [start, *start.parents]:
        if (path / 'cfgs').is_dir() and (path / 'src').is_dir():
            return path
    for candidate in [
        Path('/content/HLCV_final_project'),
        Path('/content/drive/MyDrive/HLCV_final_project'),
    ]:
        if (candidate / 'cfgs').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not find HLCV_final_project. In Colab, clone/upload the whole '
        'project folder first, then rerun this cell.'
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Using project root: {PROJECT_ROOT}')

## Available Experiments

The notebooks use the same config objects as the command-line scripts. `synthetic_debug` is the safe, fast notebook smoke test. `ftw_mask_baseline` and `ftw_dual_head` use the real FTW data under `data/ftw/` and currently restrict training to the configured country list.

In [3]:
from cfgs import field_segmentation

experiment_names = ['synthetic_debug', 'ftw_mask_baseline', 'ftw_dual_head']
summary = {
    name: {
        'model': getattr(field_segmentation, name)['model_arch'].__name__,
        'data_module': getattr(field_segmentation, name)['datamodule'].__name__,
        'data_args': getattr(field_segmentation, name)['data_args'],
        'criterion': getattr(field_segmentation, name)['criterion'].__name__,
    }
    for name in experiment_names
}
field_segmentation.FTW_COUNTRIES, summary

(['france'],
 {'synthetic_debug': {'model': 'DualHeadUNet',
   'data_module': 'SyntheticFieldDataModule',
   'data_args': {'n_samples': 96,
    'image_size': 128,
    'batch_size': 8,
    'shuffle': True,
    'heldout_split': 0.2,
    'num_workers': 0,
    'seed': 7},
   'criterion': 'DiceBCEDistancePolygonLoss'},
  'ftw_mask_baseline': {'model': 'MaskOnlyUNet',
   'data_module': 'FTWFieldDataModule',
   'data_args': {'data_dir': '/Users/daddysugar/Documents/Lectures/HLCV/HLCV_final_project/data/ftw',
    'countries': ['france'],
    'image_size': 256,
    'batch_size': 16,
    'shuffle': True,
    'max_train_samples': 8000,
    'heldout_split': 0.1,
    'num_workers': 6},
   'criterion': 'DiceBCEPolygonLoss'},
  'ftw_dual_head': {'model': 'DualHeadUNet',
   'data_module': 'FTWFieldDataModule',
   'data_args': {'data_dir': '/Users/daddysugar/Documents/Lectures/HLCV/HLCV_final_project/data/ftw',
    'countries': ['france'],
    'image_size': 256,
    'batch_size': 16,
    'shuffle': Tru

## Loss Objective

For the dual-head model, the simplified objective is:

`BCE(mask) + Dice(mask) + SmoothL1(SDF) + polygon_smoothness(mask)`

In the current code, the FTW dual-head config also enables additional SDF-gradient, boundary, and geometric regularization terms:

`BCE + Dice + distance_weight * (SDF loss + sdf_gradient_weight * SDF gradient loss) + boundary_weight * boundary loss + polygon_weight * mask smoothness`

The baseline has no SDF head, so it uses only the mask-related terms: BCE, Dice, and polygon/smoothness regularization.

## Next Step

Open `02_train_synthetic_demo.ipynb` to run a fast end-to-end smoke test. In that notebook, change `CONFIG_NAME` to `ftw_mask_baseline` or `ftw_dual_head` when you want to train on real FTW data.